In [0]:
from pyspark.sql import functions as F

bronze_path = "s3a://world-bank-api/worldbank/bronze/countries_raw_20260220_175842.json"

In [0]:
import json

# Read entire file content into driver (ok because it's tiny: ~300 countries)
raw_str = (
    spark.read
    .text(bronze_path)      # column name = "value"
    .select("value")
    .collect()[0][0]        # first row, first column
)

parsed = json.loads(raw_str)

metadata = parsed[0]       
countries = parsed[1]       # <-- list of country dicts

bronze_df = spark.createDataFrame(countries)
display(bronze_df.limit(5))
bronze_df.printSchema()


adminregion,capitalCity,id,incomeLevel,iso2Code,latitude,lendingType,longitude,name,region
"Map(id -> , iso2code -> , value -> )",Oranjestad,ABW,"Map(id -> HIC, iso2code -> XD, value -> High income)",AW,12.5167,"Map(id -> LNX, iso2code -> XX, value -> Not classified)",-70.0167,Aruba,"Map(id -> LCN, iso2code -> ZJ, value -> Latin America & Caribbean )"
"Map(id -> , iso2code -> , value -> )",,AFE,"Map(id -> NA, iso2code -> NA, value -> Aggregates)",ZH,,"Map(id -> , iso2code -> , value -> Aggregates)",,Africa Eastern and Southern,"Map(id -> NA, iso2code -> NA, value -> Aggregates)"
"Map(id -> MNA, iso2code -> XQ, value -> Middle East, North Africa, Afghanistan & Pakistan (excluding high income))",Kabul,AFG,"Map(id -> LIC, iso2code -> XM, value -> Low income)",AF,34.5228,"Map(id -> IDX, iso2code -> XI, value -> IDA)",69.1761,Afghanistan,"Map(id -> MEA, iso2code -> ZQ, value -> Middle East, North Africa, Afghanistan & Pakistan)"
"Map(id -> , iso2code -> , value -> )",,AFR,"Map(id -> NA, iso2code -> NA, value -> Aggregates)",A9,,"Map(id -> , iso2code -> , value -> Aggregates)",,Africa,"Map(id -> NA, iso2code -> NA, value -> Aggregates)"
"Map(id -> , iso2code -> , value -> )",,AFW,"Map(id -> NA, iso2code -> NA, value -> Aggregates)",ZI,,"Map(id -> , iso2code -> , value -> Aggregates)",,Africa Western and Central,"Map(id -> NA, iso2code -> NA, value -> Aggregates)"


root
 |-- adminregion: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- capitalCity: string (nullable = true)
 |-- id: string (nullable = true)
 |-- incomeLevel: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- iso2Code: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- lendingType: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- longitude: string (nullable = true)
 |-- name: string (nullable = true)
 |-- region: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [0]:
silver_df = (
    bronze_df
    .select(
        F.col("id").alias("country_code"),
        F.col("iso2Code").alias("iso2_code"),
        F.col("name").alias("country_name"),

        # nested fields
        F.col("region.id").alias("region_code"),
        F.col("region.value").alias("region_name"),

        F.col("adminregion.id").alias("admin_region_code"),
        F.col("adminregion.value").alias("admin_region_name"),

        F.col("incomeLevel.id").alias("income_level_code"),
        F.col("incomeLevel.value").alias("income_level"),

        F.col("lendingType.id").alias("lending_type_code"),
        F.col("lendingType.value").alias("lending_type"),

        F.col("capitalCity").alias("capital_city"),

        F.when(F.col("longitude") != "", F.col("longitude").cast("double")).otherwise(None).alias("longitude"),
        F.when(F.col("latitude") != "", F.col("latitude").cast("double")).otherwise(None).alias("latitude"),

        F.current_timestamp().alias("ingestion_timestamp")
    )
)

display(silver_df.limit(40))
silver_df.printSchema()


country_code,iso2_code,country_name,region_code,region_name,admin_region_code,admin_region_name,income_level_code,income_level,lending_type_code,lending_type,capital_city,longitude,latitude,ingestion_timestamp
ABW,AW,Aruba,LCN,Latin America & Caribbean,,,HIC,High income,LNX,Not classified,Oranjestad,-70.0167,12.5167,2026-02-20T21:27:39.476Z
AFE,ZH,Africa Eastern and Southern,NA,Aggregates,,,NA,Aggregates,,Aggregates,,null,null,2026-02-20T21:27:39.476Z
AFG,AF,Afghanistan,MEA,"Middle East, North Africa, Afghanistan & Pakistan",MNA,"Middle East, North Africa, Afghanistan & Pakistan (excluding high income)",LIC,Low income,IDX,IDA,Kabul,69.1761,34.5228,2026-02-20T21:27:39.476Z
AFR,A9,Africa,NA,Aggregates,,,NA,Aggregates,,Aggregates,,null,null,2026-02-20T21:27:39.476Z
AFW,ZI,Africa Western and Central,NA,Aggregates,,,NA,Aggregates,,Aggregates,,null,null,2026-02-20T21:27:39.476Z
AGO,AO,Angola,SSF,Sub-Saharan Africa,SSA,Sub-Saharan Africa (excluding high income),LMC,Lower middle income,IBD,IBRD,Luanda,13.242,-8.81155,2026-02-20T21:27:39.476Z
ALB,AL,Albania,ECS,Europe & Central Asia,ECA,Europe & Central Asia (excluding high income),UMC,Upper middle income,IBD,IBRD,Tirane,19.8172,41.3317,2026-02-20T21:27:39.476Z
AND,AD,Andorra,ECS,Europe & Central Asia,,,HIC,High income,LNX,Not classified,Andorra la Vella,1.5218,42.5075,2026-02-20T21:27:39.476Z
ARB,1A,Arab World,NA,Aggregates,,,NA,Aggregates,,Aggregates,,null,null,2026-02-20T21:27:39.476Z
ARE,AE,United Arab Emirates,MEA,"Middle East, North Africa, Afghanistan & Pakistan",,,HIC,High income,LNX,Not classified,Abu Dhabi,54.3705,24.4764,2026-02-20T21:27:39.476Z


root
 |-- country_code: string (nullable = true)
 |-- iso2_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- region_code: string (nullable = true)
 |-- region_name: string (nullable = true)
 |-- admin_region_code: string (nullable = true)
 |-- admin_region_name: string (nullable = true)
 |-- income_level_code: string (nullable = true)
 |-- income_level: string (nullable = true)
 |-- lending_type_code: string (nullable = true)
 |-- lending_type: string (nullable = true)
 |-- capital_city: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)



In [0]:
# Flag rows we *would* have dropped
silver_flagged_df = silver_df.withColumn(
    "would_be_dropped",
    (
        F.col("capital_city").isNull() |
        (F.col("capital_city") == "")
    ) &
    F.col("longitude").isNull() &
    F.col("latitude").isNull()
)

# Count
silver_flagged_df.groupBy("would_be_dropped").count().show()

# Look at some of the "dropped" ones
display(
    silver_flagged_df
      .filter("would_be_dropped = true")
      .select("country_code", "country_name", "income_level", "region_name", "capital_city", "longitude", "latitude")
      .orderBy("country_name")
)


+----------------+-----+
|would_be_dropped|count|
+----------------+-----+
|           false|  214|
|            true|   82|
+----------------+-----+



country_code,country_name,income_level,region_name,capital_city,longitude,latitude
AFR,Africa,Aggregates,Aggregates,,null,null
AFE,Africa Eastern and Southern,Aggregates,Aggregates,,null,null
AFW,Africa Western and Central,Aggregates,Aggregates,,null,null
ARB,Arab World,Aggregates,Aggregates,,null,null
CSS,Caribbean small states,Aggregates,Aggregates,,null,null
CEB,Central Europe and the Baltics,Aggregates,Aggregates,,null,null
CHI,Channel Islands,High income,Europe & Central Asia,,null,null
EAR,Early-demographic dividend,Aggregates,Aggregates,,null,null
EAS,East Asia & Pacific,Aggregates,Aggregates,,null,null
BEA,East Asia & Pacific (IBRD-only countries),Aggregates,Aggregates,,null,null


In [0]:
# Keep everything in Silver, but TAG what each row is, and whether it’s mappable. 

silver_tagged_df = (
    silver_df
    # Tag as aggregate if it looks like a group, not a country
    .withColumn(
        "is_aggregate",
        (
            (F.col("capital_city").isNull() | (F.col("capital_city") == "")) &
            F.col("income_level").contains("Aggregates")
        )
    )
    # Tag if it’s mappable (has coordinates)
    .withColumn(
        "is_mappable",
        F.col("longitude").isNotNull() & F.col("latitude").isNotNull()
    )
)


In [0]:
# Extra-Cleaning to do on the bronze data to call it silver.

silver_clean_df = (
    silver_tagged_df
    # 1. Normalize text: trim and proper-case names
    .withColumn("country_name", F.initcap(F.trim("country_name")))
    .withColumn("capital_city", F.trim("capital_city"))
    .withColumn(
        "capital_city",
        F.when(F.col("capital_city") == "", None).otherwise(F.initcap("capital_city"))
    )
    .withColumn("region_name", F.initcap(F.trim("region_name")))
    .withColumn("admin_region_name", F.initcap(F.trim("admin_region_name")))
    .withColumn("income_level", F.initcap(F.trim("income_level")))
    .withColumn("lending_type", F.initcap(F.trim("lending_type")))

    # 2. Normalize codes to upper-case

    .withColumn("country_code", F.upper(F.trim("country_code")))
    .withColumn("iso2_code", F.upper(F.trim("iso2_code")))
    .withColumn("region_code", F.upper(F.trim("region_code")))
    .withColumn("admin_region_code", F.upper(F.trim("admin_region_code")))
    .withColumn("income_level_code", F.upper(F.trim("income_level_code")))
    .withColumn("lending_type_code", F.upper(F.trim("lending_type_code")))

    # # 3. Drop obvious aggregate / non-country rows:
    # #    (rows with no capital city AND no coordinates)
    # .filter(
    #     ~(
    #         F.col("capital_city").isNull() &
    #         F.col("longitude").isNull() &
    #         F.col("latitude").isNull()
    #     )
    # )

    # # 4. Drop duplicates by country_code (just in case)
    # .dropDuplicates(["country_code"])
)

display(silver_clean_df.orderBy("country_code"))
print("Bronze total rows:", bronze_df.count())
print("Silver cleaned rows:", silver_clean_df.count())
silver_clean_df.printSchema()


country_code,iso2_code,country_name,region_code,region_name,admin_region_code,admin_region_name,income_level_code,income_level,lending_type_code,lending_type,capital_city,longitude,latitude,ingestion_timestamp,is_aggregate,is_mappable
ABW,AW,Aruba,LCN,Latin America & Caribbean,,,HIC,High Income,LNX,Not Classified,Oranjestad,-70.0167,12.5167,2026-02-20T21:27:43.638Z,false,true
AFE,ZH,Africa Eastern And Southern,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:27:43.638Z,true,false
AFG,AF,Afghanistan,MEA,"Middle East, North Africa, Afghanistan & Pakistan",MNA,"Middle East, North Africa, Afghanistan & Pakistan (excluding High Income)",LIC,Low Income,IDX,Ida,Kabul,69.1761,34.5228,2026-02-20T21:27:43.638Z,false,true
AFR,A9,Africa,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:27:43.638Z,true,false
AFW,ZI,Africa Western And Central,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:27:43.638Z,true,false
AGO,AO,Angola,SSF,Sub-saharan Africa,SSA,Sub-saharan Africa (excluding High Income),LMC,Lower Middle Income,IBD,Ibrd,Luanda,13.242,-8.81155,2026-02-20T21:27:43.638Z,false,true
ALB,AL,Albania,ECS,Europe & Central Asia,ECA,Europe & Central Asia (excluding High Income),UMC,Upper Middle Income,IBD,Ibrd,Tirane,19.8172,41.3317,2026-02-20T21:27:43.638Z,false,true
AND,AD,Andorra,ECS,Europe & Central Asia,,,HIC,High Income,LNX,Not Classified,Andorra La Vella,1.5218,42.5075,2026-02-20T21:27:43.638Z,false,true
ARB,1A,Arab World,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:27:43.638Z,true,false
ARE,AE,United Arab Emirates,MEA,"Middle East, North Africa, Afghanistan & Pakistan",,,HIC,High Income,LNX,Not Classified,Abu Dhabi,54.3705,24.4764,2026-02-20T21:27:43.638Z,false,true


Bronze total rows: 296
Silver cleaned rows: 296
root
 |-- country_code: string (nullable = true)
 |-- iso2_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- region_code: string (nullable = true)
 |-- region_name: string (nullable = true)
 |-- admin_region_code: string (nullable = true)
 |-- admin_region_name: string (nullable = true)
 |-- income_level_code: string (nullable = true)
 |-- income_level: string (nullable = true)
 |-- lending_type_code: string (nullable = true)
 |-- lending_type: string (nullable = true)
 |-- capital_city: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- is_aggregate: boolean (nullable = true)
 |-- is_mappable: boolean (nullable = false)



In [0]:
silver_path = "s3://world-bank-api/worldbank/silver/dim_country"

(
    silver_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

print("✅ Silver Delta written to:", silver_path)


silver_verify_df = spark.read.format("delta").load(silver_path)

display(silver_verify_df.limit(10))
print("Row count:", silver_verify_df.count())
silver_verify_df.printSchema()




✅ Silver Delta written to: s3://world-bank-api/worldbank/silver/dim_country


country_code,iso2_code,country_name,region_code,region_name,admin_region_code,admin_region_name,income_level_code,income_level,lending_type_code,lending_type,capital_city,longitude,latitude,ingestion_timestamp,is_aggregate,is_mappable
ABW,AW,Aruba,LCN,Latin America & Caribbean,,,HIC,High Income,LNX,Not Classified,Oranjestad,-70.0167,12.5167,2026-02-20T21:28:12.197Z,false,true
AFE,ZH,Africa Eastern And Southern,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:28:12.197Z,true,false
AFG,AF,Afghanistan,MEA,"Middle East, North Africa, Afghanistan & Pakistan",MNA,"Middle East, North Africa, Afghanistan & Pakistan (excluding High Income)",LIC,Low Income,IDX,Ida,Kabul,69.1761,34.5228,2026-02-20T21:28:12.197Z,false,true
AFR,A9,Africa,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:28:12.197Z,true,false
AFW,ZI,Africa Western And Central,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:28:12.197Z,true,false
AGO,AO,Angola,SSF,Sub-saharan Africa,SSA,Sub-saharan Africa (excluding High Income),LMC,Lower Middle Income,IBD,Ibrd,Luanda,13.242,-8.81155,2026-02-20T21:28:12.197Z,false,true
ALB,AL,Albania,ECS,Europe & Central Asia,ECA,Europe & Central Asia (excluding High Income),UMC,Upper Middle Income,IBD,Ibrd,Tirane,19.8172,41.3317,2026-02-20T21:28:12.197Z,false,true
AND,AD,Andorra,ECS,Europe & Central Asia,,,HIC,High Income,LNX,Not Classified,Andorra La Vella,1.5218,42.5075,2026-02-20T21:28:12.197Z,false,true
ARB,1A,Arab World,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:28:12.197Z,true,false
ARE,AE,United Arab Emirates,MEA,"Middle East, North Africa, Afghanistan & Pakistan",,,HIC,High Income,LNX,Not Classified,Abu Dhabi,54.3705,24.4764,2026-02-20T21:28:12.197Z,false,true


Row count: 296
root
 |-- country_code: string (nullable = true)
 |-- iso2_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- region_code: string (nullable = true)
 |-- region_name: string (nullable = true)
 |-- admin_region_code: string (nullable = true)
 |-- admin_region_name: string (nullable = true)
 |-- income_level_code: string (nullable = true)
 |-- income_level: string (nullable = true)
 |-- lending_type_code: string (nullable = true)
 |-- lending_type: string (nullable = true)
 |-- capital_city: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- is_aggregate: boolean (nullable = true)
 |-- is_mappable: boolean (nullable = true)

